In [ ]:
#Copy this and run in your bash terminal to install the required dependencies for this project
python -m venv venv
venv\Scripts\activate      # Windows
source venv/bin/activate   # Mac/Linux

In [ ]:
pip install tensorflow gradio pillow numpy matplotlib scikit-learn

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- Step 1: Load and prepare data ---
img_size = (224, 224)   # MobileNetV2's expected input size
batch_size = 16

train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    'dataset/train',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    'dataset/valid',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

num_classes = train_generator.num_classes
print("Classes found:", train_generator.class_indices)

# --- Step 2: Load MobileNetV2 (pretrained, without its original top layer) ---
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,       # exclude the original 1000-class ImageNet output layer
    weights='imagenet'       # use pretrained weights
)

# Freeze the base model - we don't want to retrain these layers yet
base_model.trainable = False

# --- Step 3: Add our own classification head ---
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# --- Step 4: Compile the model ---
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',   # same log-loss concept from Logistic Regression, extended to multiple classes
    metrics=['accuracy']
)

model.summary()

# --- Step 5: Train ---
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15
)

# --- Step 6: Save the trained model ---
model.save('model/banknote_classifier.h5')
print("Model saved successfully!")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training vs Validation Accuracy')
plt.savefig('training_plot.png')
plt.show()

In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

# --- Load the trained model ---
model = tf.keras.models.load_model('model/banknote_classifier.h5')

# --- Class labels (must match the order from training) ---
class_names = ['1000_fcfa', '2000_fcfa', '5000_fcfa', '10000_fcfa']  # update to match your actual folder order

def predict(image):
    # Preprocess the uploaded image to match training format
    image = image.resize((224, 224))
    img_array = np.array(image) / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # add batch dimension

    # Run prediction
    predictions = model.predict(img_array)[0]
    predicted_class = class_names[np.argmax(predictions)]
    confidence = float(np.max(predictions)) * 100

    return f"{predicted_class} — {confidence:.2f}% confidence"

# --- Build the Gradio interface ---
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="CFA Franc Banknote Detector",
    description="Upload a photo of a single CFA franc banknote to identify its denomination."
)

demo.launch()

In [ ]:
python app.py

In [ ]:
#The Github lines you will run in your terminal to push your project to Github
git init
git add .
git commit -m "Phase 1: Banknote classifier with MobileNetV2 and Gradio"
git branch -M main
git remote add origin <your-repo-url>
git push -u origin main